# E4 (RQ3). Robustness of the phenotypes

1. Statistical validation of each subgroup (Fisher's exact test, Holm and Benjamini-Hochberg corrections).
2. Stability with bootstrap replicates (`python bootstrap.py clinical global 100 <processes>` in `Experiments/phenotypes`).
3. Stability of the context dependence labels (bootstrap of the data, without mining again).
4. Sensitivity analyses: number of lists (k), discretization of 'hours_from_admission_to_first_icu', specimens (swabs included), cultures with only date, intermediate isolates.

The location definition cannot be analysed with this dataset ('icu_when_culture' is the ICU in which the patient resided or last resided before the culture). It is a limitation of the paper.

Outputs: `results/tables/` and, for the sensitivity analyses, `results/mining/` (candidate subgroups and DSLM output of each analysis).

In [1]:
import sys
sys.path.insert(0, "..")
import utils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from itertools import product
from subgroups.algorithms import DSLM

In [2]:
view = "clinical"
df = utils.read_mining_view(view)
# Best hyperparameters of each configuration (Experiments/phenotypes/results/clinical/best_models.csv).
best_models = pd.read_csv(utils.best_models_path(view)).set_index("configuration")

def best_subgroup_lists(configuration):
    # Output file of DSLM (subgroups library) with the best hyperparameters.
    return utils.read_subgroup_lists(utils.best_subgroup_lists_path(view, best_models, configuration))

TABLES_PATH = utils.results_path("E4-robustness", "tables")
FIGURES_PATH = utils.results_path("E4-robustness", "figures")

MINING_PATH = utils.results_path("E4-robustness", "mining")

# 1. Statistical validation of the subgroups

In [3]:
rows = []
for configuration in utils.CONFIGURATIONS:
    dataset = utils.get_configuration_dataset(df, configuration).reset_index(drop=True)
    y = utils.get_target_mask(dataset)
    for index, subgroup_list in enumerate(best_subgroup_lists(configuration)):
        for position, description in enumerate(subgroup_list):
            covered = utils.covers(dataset, description)
            table = [[(covered & y).sum(), (covered & ~y).sum()], [(~covered & y).sum(), (~covered & ~y).sum()]]
            odds_ratio, p_value = fisher_exact(table)
            rows.append({"configuration" : configuration, "list" : index + 1, "position" : position + 1, "subgroup" : utils.description_to_str(description),
                         "n" : int(covered.sum()), "precision" : y[covered].mean(), "odds_ratio" : odds_ratio, "p_value" : p_value})
validation = pd.DataFrame(rows)
validation["p_holm"] = multipletests(validation["p_value"], method="holm")[1]
validation["p_bh"] = multipletests(validation["p_value"], method="fdr_bh")[1]
validation.to_csv(TABLES_PATH / "statistical_validation.csv", index=False)
print("Significant subgroups (Holm, 0.05): " + str((validation["p_holm"] < 0.05).sum()) + "/" + str(len(validation)))
print("Significant subgroups (BH, 0.05): " + str((validation["p_bh"] < 0.05).sum()) + "/" + str(len(validation)))
validation.round(4)

Significant subgroups (Holm, 0.05): 39/45
Significant subgroups (BH, 0.05): 43/45


,configuration,list,position,subgroup,n,precision,odds_ratio,p_value,p_holm,p_bh
0,without_context,1,1,[treated_with_vancomycin_in_previous_admission...,267,0.7266,6.7087,0.0000,0.0000,0.0000
1,without_context,1,2,[admission_location = 'TRANSFER_FROM_HOSPITAL'...,168,0.6131,3.5163,0.0000,0.0000,0.0000
2,without_context,1,3,"[admission_location = 'REFERRAL', treated_with...",340,0.5853,3.4388,0.0000,0.0000,0.0000
3,without_context,1,4,"[admission_location = 'REFERRAL', previous_iso...",738,0.3672,1.2573,0.0136,0.0679,0.0149
4,without_context,2,1,[previous_exposure_to_antifungal_agents = 'yes...,133,0.7820,8.1469,0.0000,0.0000,0.0000
5,without_context,2,2,[previous_exposure_to_antifungal_agents = 'no'...,854,0.5199,3.5610,0.0000,0.0000,0.0000
6,without_context,2,3,"[patient_age = 'ADULT', previous_exposure_to_t...",947,0.3654,1.2813,0.0044,0.0400,0.0053
7,without_context,2,4,"[admission_location = 'REFERRAL', organ_transp...",730,0.3658,1.2447,0.0195,0.0685,0.0204
8,without_context,3,1,[previous_exposure_to_antifungal_agents = 'yes...,160,0.7562,7.1736,0.0000,0.0000,0.0000
9,without_context,3,2,[admission_location = 'TRANSFER_FROM_HOSPITAL'...,204,0.6373,4.0262,0.0000,0.0000,0.0000


# 2. Stability of the phenotypes (bootstrap)

For each subgroup of the best lists: the frequency with which the same description appears in the replicates, and the mean of the best Jaccard similarity (instances covered in the original dataset) between the subgroup and the subgroups of each replicate.

In [4]:
bootstrap_path = utils.bootstrap_path(view)
stability_rows = []
for configuration in [c for c in utils.CONFIGURATIONS if (bootstrap_path / c).exists()]:
    dataset = utils.get_configuration_dataset(df, configuration).reset_index(drop=True)
    replicates = [utils.read_subgroup_lists(f) for f in sorted((bootstrap_path / configuration).glob("b_*.txt"))]
    replicate_descriptions = [set(d for sl in r for d in sl) for r in replicates]
    replicate_masks = [[utils.covers(dataset, d) for d in descriptions] for descriptions in replicate_descriptions]
    for index, subgroup_list in enumerate(best_subgroup_lists(configuration)):
        for position, description in enumerate(subgroup_list):
            mask = utils.covers(dataset, description)
            jaccards = [max([(mask & m).sum() / max((mask | m).sum(), 1) for m in masks], default=0) for masks in replicate_masks]
            stability_rows.append({"configuration" : configuration, "list" : index + 1, "position" : position + 1, "subgroup" : utils.description_to_str(description),
                                   "replicates" : len(replicates), "frequency" : np.mean([description in r for r in replicate_descriptions]), "mean_best_jaccard" : np.mean(jaccards)})
stability = pd.DataFrame(stability_rows)
stability.to_csv(TABLES_PATH / "phenotype_stability.csv", index=False)
stability.round(3)

,configuration,list,position,subgroup,replicates,frequency,mean_best_jaccard
0,without_context,1,1,[treated_with_vancomycin_in_previous_admission...,100,0.03,0.794
1,without_context,1,2,[admission_location = 'TRANSFER_FROM_HOSPITAL'...,100,0.25,0.722
2,without_context,1,3,"[admission_location = 'REFERRAL', treated_with...",100,0.15,0.707
3,without_context,1,4,"[admission_location = 'REFERRAL', previous_iso...",100,0.19,0.837
4,without_context,2,1,[previous_exposure_to_antifungal_agents = 'yes...,100,0.32,0.779
5,without_context,2,2,[previous_exposure_to_antifungal_agents = 'no'...,100,0.08,0.698
6,without_context,2,3,"[patient_age = 'ADULT', previous_exposure_to_t...",100,0.13,0.777
7,without_context,2,4,"[admission_location = 'REFERRAL', organ_transp...",100,0.12,0.826
8,without_context,3,1,[previous_exposure_to_antifungal_agents = 'yes...,100,0.08,0.761
9,without_context,3,2,[admission_location = 'TRANSFER_FROM_HOSPITAL'...,100,0.27,0.765


In [5]:
stability.groupby("configuration")[["frequency", "mean_best_jaccard"]].mean().round(3)

,frequency,mean_best_jaccard
configuration,,
with_context,0.159,0.714
without_context,0.140,0.747


# 3. Stability of the context dependence labels

Bootstrap of the instances (B = 200), recomputing the context dependence measure and the labels (with the Benjamini-Hochberg correction in each replicate) of the phenotypes of E2. The stability of a phenotype is the proportion of replicates with the same label as in the original dataset.

In [6]:
original = pd.read_csv(utils.results_path("E2-context-dependence", "tables") / "context_dependence.csv")
descriptions = [utils.str_to_description(d) for d in original["description"]]
rng = np.random.default_rng(utils.RANDOM_SEED)
labels = []
for b in range(200):
    sample = df.iloc[rng.integers(0, len(df), len(df))].reset_index(drop=True)
    replicate = utils.label_context_dependence(pd.DataFrame([utils.context_dependence(sample, d) for d in descriptions]))
    labels.append(replicate["label"].values)
labels = np.array(labels)
pd.DataFrame(labels, columns=original["description"]).to_csv(TABLES_PATH / "bootstrap_labels.csv", index_label="replicate")
original["label_stability"] = (labels == original["label"].to_numpy(dtype=object)[np.newaxis, :]).mean(axis=0)
original[["description", "label", "i2", "label_stability"]].to_csv(TABLES_PATH / "label_stability.csv", index=False)
original[["description", "label", "i2", "label_stability"]].sort_values("label_stability").round(3)

,description,label,i2,label_stability
25,[previous_isolation_of_pseudomonas_aeruginosa ...,TRANSVERSAL,0.570,0.425
26,"[patient_age = 'ADULT', previous_exposure_to_c...",TRANSVERSAL,0.000,0.520
13,[organ_transplant_in_previous_admissions = 'no...,NOT_ASSOCIATED,0.000,0.530
21,"[patient_age = 'ADULT', previous_isolation_of_...",TRANSVERSAL,0.000,0.555
18,[previous_exposure_to_antifungal_agents = 'no'...,TRANSVERSAL,0.412,0.565
5,[previous_exposure_to_antifungal_agents = 'no'...,TRANSVERSAL,0.412,0.565
20,[previous_exposure_to_antifungal_agents = 'no'...,TRANSVERSAL,0.412,0.565
11,[treated_with_vancomycin_in_previous_admission...,TRANSVERSAL,0.414,0.570
6,"[patient_age = 'ADULT', previous_exposure_to_t...",TRANSVERSAL,0.000,0.595
29,[patient_gender = 'F'],NOT_ASSOCIATED,0.000,0.600


In [7]:
original.groupby("label")["label_stability"].agg(["count", "mean"]).round(3)

,count,mean
label,,
CONTEXT_DRIVEN,2,0.938
NOT_ASSOCIATED,2,0.565
SPACE_MODULATED,5,0.754
SPACE_SPECIFIC,4,0.762
TRANSVERSAL,24,0.737


# 4. Sensitivity analyses

In [8]:
def mine(dataset, name, configuration_of_parameters, maximum_number_of_subgroup_lists=utils.MAXIMUM_NUMBER_OF_SUBGROUP_LISTS):
    row = best_models.loc[configuration_of_parameters]
    dataset = dataset.reset_index(drop=True)
    candidates = MINING_PATH / (name + "_candidates.txt")
    lists = MINING_PATH / (name + "_lists.txt")
    utils.generate_candidate_subgroups(dataset, utils.TARGET, candidates)
    # DSLM algorithm of the subgroups library (diverse top-k subgroup lists).
    model = DSLM(input_file_path = str(candidates),
                 max_sl = maximum_number_of_subgroup_lists,
                 sl_max_size = utils.MAXIMUM_NUMBER_OF_SUBGROUPS_PER_SUBGROUP_LIST,
                 beta = float(row["beta"]),
                 maximum_positive_overlap = float(row["maximum_positive_overlap"]),
                 maximum_negative_overlap = float(row["maximum_negative_overlap"]),
                 output_file_path = str(lists))
    model.fit(dataset, utils.TARGET)
    subgroup_lists = utils.read_subgroup_lists(lists)
    measures = utils.subgroup_lists_quality(dataset, subgroup_lists)
    measures["analysis"] = name
    measures["instances"] = len(dataset)
    return measures, subgroup_lists

def print_lists(subgroup_lists):
    for index, subgroup_list in enumerate(subgroup_lists):
        print("## Subgroup list " + str(index + 1) + " ##")
        for position, description in enumerate(subgroup_list):
            print("  s" + str(position + 1) + ": " + utils.description_to_str(description))

## 4.1. Number of subgroup lists (k)

In [9]:
rows = []
for configuration, k in product(["without_context", "with_context"], [1, 3, 5]):
    measures, _ = mine(utils.get_configuration_dataset(df, configuration), configuration + "_k" + str(k), configuration, maximum_number_of_subgroup_lists=k)
    rows.append(measures)
sensitivity_k = pd.DataFrame(rows).set_index("analysis")
sensitivity_k.to_csv(TABLES_PATH / "sensitivity_number_of_lists.csv")
sensitivity_k.round(3)

,number_of_lists,number_of_subgroups,mean_subgroups_per_list,quality,coverage,diversity,context_proportion,instances
analysis,,,,,,,,
without_context_k1,1,4,4.000,0.106,0.437,NaN,0.000,2505
without_context_k3,3,14,4.667,0.126,0.746,0.738,0.000,2505
without_context_k5,5,24,4.800,0.130,0.822,0.762,0.000,2505
with_context_k1,1,5,5.000,0.119,0.538,NaN,0.600,2505
with_context_k3,3,16,5.333,0.120,0.694,0.611,0.562,2505
with_context_k5,5,26,5.200,0.120,0.803,0.687,0.615,2505


## 4.2. Discretization of 'hours_from_admission_to_first_icu' (fixed clinical cut points: 24 and 72 hours)

In [10]:
auxiliary = utils.read_auxiliary_data(view)
hours = auxiliary["hours_from_admission_to_first_icu_raw"]
alternative = df.copy()
alternative["hours_from_admission_to_first_icu"] = pd.Series(np.digitize(hours, [-0.5, 0.0, 24, 72], right=True)).map({0 : "NO_ICU_BEFORE_CULTURE", 1 : "0", 2 : "0<x<=24", 3 : "24<x<=72", 4 : "72<x"}).values
measures_alternative, lists_alternative = mine(alternative, "with_context_alternative_discretization", "with_context")
measures_original = utils.subgroup_lists_quality(df, best_subgroup_lists("with_context"))
print("Original discretization: " + str({k : round(v, 3) for k, v in measures_original.items()}))
print("Alternative discretization: " + str({k : round(v, 3) if isinstance(v, float) else v for k, v in measures_alternative.items()}))
measures_original["analysis"] = "with_context_original_discretization"
pd.DataFrame([measures_original, measures_alternative]).set_index("analysis").to_csv(TABLES_PATH / "sensitivity_discretization.csv")
print_lists(lists_alternative)

Original discretization: {'number_of_lists': 3, 'number_of_subgroups': 16, 'mean_subgroups_per_list': 5.333, 'quality': 0.12, 'coverage': np.float64(0.694), 'diversity': 0.611, 'context_proportion': 0.562}
Alternative discretization: {'number_of_lists': 3, 'number_of_subgroups': 16, 'mean_subgroups_per_list': 5.333, 'quality': 0.12, 'coverage': np.float64(0.694), 'diversity': 0.611, 'context_proportion': 0.562, 'analysis': 'with_context_alternative_discretization', 'instances': 2505}
## Subgroup list 1 ##
  s1: [treated_with_vancomycin_in_previous_admissions = 'yes', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s2: [service_when_culture = 'OMED', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s3: [readmission = 'no', service_when_culture = 'MED', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s4: [admission_type = 'EMERGENCY', service_when_culture = 'SURG', treated_with_vancomycin_in_the_current_admis

## 4.3. Specimens: clinical cultures and swabs

In [11]:
df_all = utils.read_mining_view("all-specimens")
rows = []
for configuration in ["without_context", "with_context"]:
    measures, subgroup_lists = mine(utils.get_configuration_dataset(df_all, configuration), "all_specimens_" + configuration, configuration)
    measures["proportion_of_subgroups_with_swab"] = np.mean([("culture_specimen_type", "SWAB") in d for sl in subgroup_lists for d in sl])
    rows.append(measures)
    print("#### " + configuration + " ####")
    print_lists(subgroup_lists)
sensitivity_specimens = pd.DataFrame(rows).set_index("analysis")
sensitivity_specimens.to_csv(TABLES_PATH / "sensitivity_all_specimens.csv")
sensitivity_specimens.round(3)

#### without_context ####
## Subgroup list 1 ##
  s1: [culture_specimen_type = 'SWAB', treated_with_vancomycin_in_previous_admissions = 'yes']
  s2: [culture_specimen_type = 'SWAB', treated_with_vancomycin_in_previous_admissions = 'no', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s3: [previous_exposure_to_fluorquinolones = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s4: [admission_location = 'TRANSFER_FROM_HOSPITAL', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s5: [admission_location = 'TRANSFER_FROM_HOSPITAL', previous_isolation_of_pseudomonas_aeruginosa = 'no']
  s6: [patient_age = 'ADULT']
## Subgroup list 2 ##
  s1: [culture_specimen_type = 'SWAB', previous_exposure_to_fluorquinolones = 'yes']
  s2: [culture_specimen_type = 'SWAB', previous_exposure_to_fluorquinolones = 'no', treated_with_vancomycin_in_the_current_admission_befo

#### with_context ####
## Subgroup list 1 ##
  s1: [culture_specimen_type = 'SWAB', treated_with_vancomycin_in_previous_admissions = 'yes']
  s2: [culture_specimen_type = 'SWAB', treated_with_vancomycin_in_previous_admissions = 'no', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s3: [previous_exposure_to_fluorquinolones = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s4: [icu_when_culture = 'MICU', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s5: [icu_when_culture = 'MICU', previous_isolation_of_erv = 'no']
  s6: [patient_age = 'ADULT']
## Subgroup list 2 ##
  s1: [culture_specimen_type = 'SWAB', readmission = 'yes', service_when_culture = 'MED']
  s2: [culture_specimen_type = 'SWAB', hours_from_admission_to_first_icu = '83<x', treated_with_vancomycin_in_the_current_admission_before_culture = 'yes']
  s3: [previous_exposure_to_antifungal

,number_of_lists,number_of_subgroups,mean_subgroups_per_list,quality,coverage,diversity,context_proportion,instances,proportion_of_subgroups_with_swab
analysis,,,,,,,,,
all_specimens_without_context,3,14,4.667,0.128,0.643,0.572,0.000,3119,0.429
all_specimens_with_context,3,19,6.333,0.161,0.769,0.721,0.632,3119,0.421


## 4.4. Cultures with only date and intermediate isolates

In [12]:
rows = []
filters = {
    "without_date_only_cultures" : ~auxiliary["culture_date_only"].values,
    "without_intermediate_isolates" : ~auxiliary["intermediate"].values,
}
for name, mask in filters.items():
    for configuration in ["without_context", "with_context"]:
        measures, _ = mine(utils.get_configuration_dataset(df[mask], configuration), name + "_" + configuration, configuration)
        rows.append(measures)
for configuration in ["without_context", "with_context"]:
    measures = utils.subgroup_lists_quality(utils.get_configuration_dataset(df, configuration).reset_index(drop=True), best_subgroup_lists(configuration))
    measures["analysis"] = "original_" + configuration
    measures["instances"] = len(df)
    rows.append(measures)
sensitivity_filters = pd.DataFrame(rows).set_index("analysis")
sensitivity_filters.to_csv(TABLES_PATH / "sensitivity_date_only_and_intermediate.csv")
sensitivity_filters.round(3)

,number_of_lists,number_of_subgroups,mean_subgroups_per_list,quality,coverage,diversity,context_proportion,instances
analysis,,,,,,,,
without_date_only_cultures_without_context,3,11,3.667,0.110,0.734,0.839,0.000,2265
without_date_only_cultures_with_context,3,14,4.667,0.109,0.692,0.646,0.714,2265
without_intermediate_isolates_without_context,3,16,5.333,0.141,0.818,0.735,0.000,2494
without_intermediate_isolates_with_context,3,16,5.333,0.118,0.794,0.794,0.625,2494
original_without_context,3,14,4.667,0.126,0.746,0.738,0.000,2505
original_with_context,3,16,5.333,0.120,0.694,0.611,0.562,2505


## 4.5. Without 'treated_with_vancomycin_in_the_current_admission_before_culture'

In the query, this attribute compares PRESCRIPTIONS.STARTDATE (date without time) with the culture time, so it can include vancomycin started on the same day after the culture (empirical treatment). Since this attribute dominates the phenotypes, we check the results without it.

In [13]:
attribute = "treated_with_vancomycin_in_the_current_admission_before_culture"
rows = []
for configuration in ["without_context", "with_context"]:
    measures, subgroup_lists = mine(utils.get_configuration_dataset(df.drop(columns=[attribute]), configuration), "without_current_vancomycin_" + configuration, configuration)
    rows.append(measures)
    print("#### " + configuration + " ####")
    print_lists(subgroup_lists)
sensitivity_vancomycin = pd.DataFrame(rows).set_index("analysis")
sensitivity_vancomycin.to_csv(TABLES_PATH / "sensitivity_without_current_vancomycin.csv")
sensitivity_vancomycin.round(3)

#### without_context ####
## Subgroup list 1 ##
  s1: [previous_exposure_to_antifungal_agents = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes']
  s2: [previous_exposure_to_antifungal_agents = 'no', treated_with_vancomycin_in_previous_admissions = 'yes']
## Subgroup list 2 ##
  s1: [previous_exposure_to_fluorquinolones = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes']
## Subgroup list 3 ##
  s1: [admission_type = 'EMERGENCY', treated_with_vancomycin_in_previous_admissions = 'yes']


#### with_context ####
## Subgroup list 1 ##
  s1: [service_when_culture = 'OMED']
  s2: [service_when_culture = 'MED', treated_with_vancomycin_in_previous_admissions = 'yes']
  s3: [hours_from_admission_to_first_icu = '83<x']
  s4: [previous_exposure_to_fluorquinolones = 'yes', previous_isolation_of_enterobacterales_cre = 'no', treated_with_vancomycin_in_previous_admissions = 'yes']
  s5: [admission_location = 'REFERRAL']
## Subgroup list 2 ##
  s1: [previous_exposure_to_antifungal_agents = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes']
  s2: [previous_exposure_to_antifungal_agents = 'no', treated_with_vancomycin_in_previous_admissions = 'yes']
## Subgroup list 3 ##
  s1: [admission_type = 'EMERGENCY', previous_exposure_to_fluorquinolones = 'yes', treated_with_vancomycin_in_previous_admissions = 'yes']


,number_of_lists,number_of_subgroups,mean_subgroups_per_list,quality,coverage,diversity,context_proportion,instances
analysis,,,,,,,,
without_current_vancomycin_without_context,3,4,1.333,0.048,0.190,0.722,0.000,2505
without_current_vancomycin_with_context,3,8,2.667,0.068,0.475,0.813,0.375,2505
